In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = "/content/drive/My Drive/ai_programming/final/"
MODEL_PATH = BASE_PATH + "model/"

Mounted at /content/drive


In [ ]:
# Step 1 — Download and locate dataset
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("arjunverma2004/ai-vs-human-text-balanced-180k-records")

print(path)
print(os.listdir(path))

100%|██████████| 251M/251M [00:03<00:00, 80.6MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/arjunverma2004/ai-vs-human-text-balanced-180k-records/versions/1
['AI_Human_balanced_dataset.csv']


In [ ]:
# Step 2 Load dataset and inspect structure
# Find CSV file automatically in dataset folder
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

# Check if any CSV exists
if not csv_files:
    raise FileNotFoundError("No CSV file found in dataset folder.")

# Use the first CSV file found
file_path = os.path.join(path, csv_files[0])

print("Using file:", file_path)

# Load dataset
df = pd.read_csv(file_path)

# Preview
display(df.head())
print("\nColumns:", df.columns)

Using file: /root/.cache/kagglehub/datasets/arjunverma2004/ai-vs-human-text-balanced-180k-records/versions/1/AI_Human_balanced_dataset.csv


,text,generated
0,Do curfews keep teenagers from Getting into tr...,0.0
1,"In this article ""The Challenge of Exploring Ve...",0.0
2,With THP rapid growth of THP Internet in recen...,0.0
3,The electoral College is the way Us United Sta...,0.0
4,This technology of you can calculate the emoti...,0.0



Columns: Index(['text', 'generated'], dtype='object')


In [ ]:
# Step 3 — Standardize columns
# Rename column
df = df.rename(columns={"generated": "label"})

# Ensure label is numeric (0 = human, 1 = AI)
df['label'] = df['label'].astype(int)

# Verify
print(df[['text', 'label']].head())
print("\nLabel distribution:")
print(df['label'].value_counts())

                                                text  label
0  Do curfews keep teenagers from Getting into tr...      0
1  In this article "The Challenge of Exploring Ve...      0
2  With THP rapid growth of THP Internet in recen...      0
3  The electoral College is the way Us United Sta...      0
4  This technology of you can calculate the emoti...      0

Label distribution:
label
0    181438
1    181438
Name: count, dtype: int64


In [ ]:
# Step 4 Cleaning
# Remove missing values
df = df.dropna()

# Strip whitespace
df['text'] = df['text'].str.strip()

# Remove empty rows
df = df[df['text'] != ""]

In [ ]:
# Step 5 Train / validation split
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.1,
    random_state=42
)

In [ ]:
print("Sample text:", train_texts.iloc[0])
print("Sample label:", train_labels.iloc[0])

Sample text: Advantages and disadvantages of working either in a group or alone exist and should be considered when deciding on collaborative efforts. Working in a group provides the potential for enhanced creativity and the synergy of multiple minds focused on the same task. This can result in improved accuracy, insight, and production. Working in a group is often more enjoyable and provides a sense of camaraderie between the members. On the other hand, working alone offers the individual the ability to focus and attempt tasks methodically while benefiting from their own unique perspective or skill set. Working alone may also be better for achieving more sensitive or private goals. Ultimately, the decision on if one should work in a group or alone must be considered carefully and will depend on the specific project, desired outcome, and unique situation.
Sample label: 1


In [ ]:
# Step 6 Create Dataset class (for Trainer)
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# Step 7 Tokenization (batched to avoid RAM issues)

from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Reduce dataset size
train_texts = train_texts[:50000]
train_labels = train_labels[:50000]

val_texts = val_texts[:10000]
val_labels = val_labels[:10000]

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")


# Batch tokenization function
def batch_tokenize(texts, tokenizer, batch_size=1000):
    all_encodings = {
        "input_ids": [],
        "attention_mask": []
    }

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        encodings = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=128
        )

        all_encodings["input_ids"].extend(encodings["input_ids"])
        all_encodings["attention_mask"].extend(encodings["attention_mask"])

        # Progress indicator
        if i % 5000 == 0:
            print(f"Processed {i} samples...")

    return all_encodings


# Run tokenization
print("Starting tokenization...")

train_encodings = batch_tokenize(list(train_texts), tokenizer)
val_encodings = batch_tokenize(list(val_texts), tokenizer)

print("Tokenization complete!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Training samples: 50000
Validation samples: 10000
Starting tokenization...
Processed 0 samples...
Processed 5000 samples...
Processed 10000 samples...
Processed 15000 samples...
Processed 20000 samples...
Processed 25000 samples...
Processed 30000 samples...
Processed 35000 samples...
Processed 40000 samples...
Processed 45000 samples...
Processed 0 samples...
Processed 5000 samples...
Tokenization complete!


In [ ]:
# Step 8 — Dataset class

import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


# Create dataset objects
train_dataset = TextDataset(train_encodings, train_labels)
val_dataset = TextDataset(val_encodings, val_labels)

In [ ]:
# Step 9 — Load model

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.eval()  # good practice

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
# Step 10 — Training setup

from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,                 # keep small for speed
    per_device_train_batch_size=8,      # safer for Colab memory
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    save_strategy="no",                 # avoid extra files
    report_to="none"                   # disable wandb warnings
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# Step 11 — Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Step 12 — Train 🚀

print("Starting training...")
trainer.train()


# Step 13 — Evaluate

print("Evaluating model...")
results = trainer.evaluate()

print("Evaluation results:")
print(results)

Starting training...


Step,Training Loss
500,0.244306
1000,0.155533
1500,0.122628
2000,0.086359
2500,0.078535
3000,0.064975


Step,Training Loss
500,0.244306
1000,0.155533
1500,0.122628
2000,0.086359
2500,0.078535
3000,0.064975
3500,0.060770
4000,0.054890
4500,0.049036
5000,0.034347


Evaluating model...


Evaluation results:
{'eval_loss': 0.03662775084376335, 'eval_runtime': 36.4365, 'eval_samples_per_second': 274.45, 'eval_steps_per_second': 34.306, 'epoch': 1.0}


In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)

accuracy = accuracy_score(val_labels, preds)
print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.9917


In [ ]:
MODEL_PATH = "/content/drive/My Drive/ai_programming/final/model/"

model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print("Model saved to Google Drive!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Google Drive!
